# 02. Preprocessamento: Split e Baseline

A limpeza dos dados, incluindo a remoção de `idade` implausível, já foi decidida e aplicada no notebook 01, com o teste de sensibilidade documentado lá. Este notebook parte do `dataset_limpo.csv` já tratado e cuida só do split e do baseline.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src import data_prep, modeling, evaluation, interpretability, visualization

visualization.set_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

In [2]:
df = pd.read_csv(PROCESSED_DIR / "dataset_limpo.csv")
target = data_prep.TARGET
X, y = data_prep.get_feature_target(df, target)
print("Formato do dataset limpo (idade implausível já removida no notebook 01):", df.shape)
df.head()

Formato do dataset limpo (idade implausível já removida no notebook 01): (2072, 10)


,idade,renda,tempo_cliente,frequencia,reclamacoes,engajamento,dias_sem_comprar,desconto,ticket,gasto_mensal
0,35.854694,4379.599275,5.100554,8.366152,2,99.646007,9.417171,11.941011,174.279971,2659.349831
1,25.207447,5405.197007,6.223224,9.449941,3,99.483219,23.085392,19.219608,223.815191,3546.329801
2,38.940057,4299.615577,2.903725,7.726809,2,98.699483,7.986427,16.548449,171.451762,3032.314364
3,60.551034,5995.765751,5.490279,15.457916,1,98.536292,23.180922,18.463324,109.668871,3941.132663
4,35.963310,4325.959318,3.231117,11.895505,2,98.339928,30.608135,10.958427,207.152144,2390.319692


## 3. Criar o split

Como não há estrutura temporal nem clientes repetidos entre as linhas (cada linha representa um cliente diferente), um holdout aleatório de 80/20 é apropriado aqui, sem risco de vazamento por dependência temporal ou de grupo. Fixamos o `random_state` para o split ser reprodutível.

Vale lembrar que essa justificativa depende da suposição, já declarada no README e no notebook 01, de que a base não tem estrutura temporal. Essa suposição não pode ser confirmada sem um dicionário de dados oficial da fonte (ver `DATA_DICTIONARY.md`).

In [3]:
X_train, X_test, y_train, y_test = data_prep.make_split(X, y)

print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")
print(f"Média do gasto no treino: {y_train.mean():.1f} | no teste: {y_test.mean():.1f}")

Treino: 1657 linhas | Teste: 415 linhas
Média do gasto no treino: 2388.3 | no teste: 2366.3


## 4. Criar o baseline

Criamos uma referência mínima, que prevê sempre a média do gasto visto no treino para qualquer cliente do teste. Um modelo só faz sentido se conseguir superar esse baseline.

In [4]:
baseline = modeling.get_baseline()
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

baseline_metrics = evaluation.regression_metrics(y_test, pred_baseline)
print(f"Baseline (média do treino = R$ {baseline.constant_[0][0]:.1f})")
for k, v in baseline_metrics.items():
    print(f"  {k} = {v:.3f}")

Baseline (média do treino = R$ 2388.3)
  MAE = 418.924
  RMSE = 520.454
  R2 = -0.002
  MAPE = 0.198


Um R² próximo de zero, ou até negativo, é esperado aqui: por definição, prever sempre a média do treino não explica quase nada da variância fora da amostra. O MAPE do baseline, em torno de 20%, serve de referência para comparar com o MAPE do modelo final no notebook 06.

## Saída desta etapa

In [5]:
pd.concat([X_train, y_train], axis=1).to_csv(PROCESSED_DIR / "train.csv", index=False)
pd.concat([X_test, y_test], axis=1).to_csv(PROCESSED_DIR / "test.csv", index=False)
print("train.csv e test.csv salvos em", PROCESSED_DIR)

train.csv e test.csv salvos em /tmp/project/data/processed
